In [1]:
%matplotlib qt5

import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.vector import Miller, Vector3d
from orix.plot import IPFColorKeyTSL
from matplotlib.lines import Line2D
from dataclasses import dataclass

In [2]:
## Paths used for loading and saving data
path = Path("D:\\Datasets\\SPED_tilt_series\\SPED_tilt_series")
path_calibrated = Path("D:\\Datasets\\SPED_tilt_series\\series_calibrated")
path_target = Path("D:\\Datasets\\SPED_tilt_series\\target_areas")
path_reference = Path("D:\\Datasets\\SPED_tilt_series\\reference_areas")

In [ ]:
# The scaling factor used was determined by manually calibrating one of the datasets and extracting the scale factor from the calibrated data. #
def preprocess_data(path, path_calibrated, path_target, path_reference, scale_factor):
    files = sorted(path.glob("*.hspy"))
    for i in files:
        filename = i.stem
        number = filename.rsplit("_", 1)[-1]

        ## Calibrate the data ##
        load_i = hs.load(i)
        load_i.axes_manager[2].scale = scale_factor
        load_i.axes_manager[3].scale = scale_factor

        shifts = load_i.get_direct_beam_position(method="blur", sigma=5, half_square_width=10)
        linear_shifts = shifts.get_linear_plane()
        load_i_centered = load_i.center_direct_beam(shifts=linear_shifts, inplace=False)


        ## Select region of interest ##
        reference_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)
        target_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)

        load_i_centered.plot()

        reference_roi = reference_rectangle.interactive(load_i_centered)
        target_roi = target_rectangle.interactive(load_i_centered, color='C1')
        
        plt.show(block=True)

        reference_roi.save(path_reference / f"SPED_tilt_series_reference_{number}.hspy")
        target_roi.save(path_target / f"SPED_tilt_series_target_{number}.hspy")
        load_i_centered.save(path_calibrated / f"SPED_tilt_series_calibrated_{number}.hspy")

In [3]:
# Define the crystal structure for aluminum (Al)
a = 4.05
atoms = [Atom('Al', [0, 0, 0]), Atom('Al', [0.5, 0.5, 0]), Atom('Al', [0.5, 0, 0.5]), Atom('Al', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Al', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [ ]:
target_files = sorted(path_target.glob("*.hspy"))
reference_files = sorted(path_reference.glob("*.hspy"))

for target_file, reference_file in zip(target_files, reference_files):
    target_data = hs.load(target_file)
    reference_data = hs.load(reference_file)

    target_sum = target_data.sum()
    reference_sum = reference_data.sum()

    target_sum = 


c:\Users\denis\HyperSpy-bundle\Lib\site-packages\orix\plot\stereographic_plot.py:231: UserWarning: You passed both c and facecolor/facecolors for the markers. c has precedence over facecolor/facecolors. This behavior may change in the future.
  super().scatter(x, y, c=c, s=s, **updated_kwargs)
